# Feature Engineering for Transfer Learning

## Pre-trained Model Feature Extraction

This notebook demonstrates **Transfer Learning** techniques for feature extraction using pre-trained models, covering:

- **Pre-trained CNN Feature Extraction** - Using ResNet, VGG, EfficientNet
- **Fine-tuning Strategies** - Layer-wise fine-tuning approaches  
- **Feature Fusion Techniques** - Combining features from multiple layers
- **Domain Adaptation** - Adapting pre-trained features to new domains
- **Multi-scale Feature Extraction** - Features at different resolutions
- **Knowledge Distillation** - Teacher-student model approaches

### 🔧 Applications:
- **Computer Vision**: Image classification, object detection
- **Medical Imaging**: Disease diagnosis, anomaly detection
- **Industrial IoT**: Quality control, predictive maintenance
- **Time Series**: Signal analysis using CNN-based approaches
- **Multimodal Learning**: Cross-domain feature learning

### Pre-trained Models Used:
- **ResNet Family**: ResNet18, ResNet50, ResNet152
- **EfficientNet**: EfficientNet-B0 to B7
- **Vision Transformer**: ViT-Base, ViT-Large
- **MobileNet**: Lightweight mobile-optimized features
- **DenseNet**: Densely connected convolutional networks

---

In [ ]:
# Essential Imports and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Device configuration with fallback
def setup_device_with_memory_check():
    """Setup device with memory optimization"""
    if torch.cuda.is_available():
        device = torch.device('cuda')
        # Clear cache and check memory
        torch.cuda.empty_cache()
        memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"CUDA available: {torch.cuda.get_device_name()}")
        print(f"GPU Memory: {memory_gb:.1f} GB")
        
        # Enable mixed precision if available
        if hasattr(torch.cuda.amp, 'GradScaler'):
            print("🔧 Mixed precision training available")
            
    else:
        device = torch.device('cpu')
        print("⚠️ CUDA not available, using CPU")
        print("Note: Pre-trained models will run slower on CPU")
    
    return device

device = setup_device_with_memory_check()

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print(f"🔧 PyTorch version: {torch.__version__}")
print(f"Notebook: Transfer Learning Feature Engineering")
print("=" * 70)

# Create Synthetic Image-like Data for Transfer Learning
def generate_synthetic_image_data(num_samples=5000, image_size=224):
    """Generate synthetic data that mimics natural images"""
    
    # Create different types of synthetic "images"
    images = []
    labels = []
    
    categories = {
        'geometric_patterns': 0,
        'noise_patterns': 1,
        'gradient_patterns': 2,
        'texture_patterns': 3,
        'mixed_patterns': 4
    }
    
    samples_per_category = num_samples // len(categories)
    
    for category, label in categories.items():
        for i in range(samples_per_category):
            if category == 'geometric_patterns':
                # Create geometric patterns (circles, squares)
                img = create_geometric_pattern(image_size)
            elif category == 'noise_patterns':
                # Create structured noise
                img = create_noise_pattern(image_size)
            elif category == 'gradient_patterns':
                # Create gradient patterns
                img = create_gradient_pattern(image_size)
            elif category == 'texture_patterns':
                # Create texture-like patterns
                img = create_texture_pattern(image_size)
            else:
                # Mixed patterns
                img = create_mixed_pattern(image_size)
            
            images.append(img)
            labels.append(label)
    
    return np.array(images), np.array(labels)

def create_geometric_pattern(size):
    """Create geometric patterns"""
    img = np.zeros((3, size, size))
    center = size // 2
    
    # Random geometric shape
    shape_type = np.random.choice(['circle', 'square', 'triangle'])
    
    if shape_type == 'circle':
        y, x = np.ogrid[:size, :size]
        mask = (x - center)**2 + (y - center)**2 <= (size//4)**2
        img[:, mask] = np.random.rand(3, 1)
    elif shape_type == 'square':
        start = size//4
        end = 3*size//4
        img[:, start:end, start:end] = np.random.rand(3, 1, 1)
    else:  # triangle
        for i in range(size):
            for j in range(size):
                if i + j > size and abs(i - j) < size//4:
                    img[:, i, j] = np.random.rand(3)
    
    return img

def create_noise_pattern(size):
    """Create structured noise patterns"""
    img = np.random.rand(3, size, size)
    
    # Add some structure
    frequency = np.random.uniform(0.1, 0.5)
    x = np.linspace(0, frequency * 2 * np.pi, size)
    y = np.linspace(0, frequency * 2 * np.pi, size)
    X, Y = np.meshgrid(x, y)
    
    pattern = np.sin(X) * np.cos(Y)
    img[0] *= (1 + 0.5 * pattern)
    img[1] *= (1 + 0.5 * np.sin(2*X))
    img[2] *= (1 + 0.5 * np.cos(2*Y))
    
    return np.clip(img, 0, 1)

def create_gradient_pattern(size):
    """Create gradient patterns"""
    img = np.zeros((3, size, size))
    
    # Random gradient direction
    direction = np.random.choice(['horizontal', 'vertical', 'diagonal', 'radial'])
    
    if direction == 'horizontal':
        gradient = np.linspace(0, 1, size)
        img[0] = gradient[None, :]
        img[1] = gradient[None, :] ** 2
        img[2] = 1 - gradient[None, :]
    elif direction == 'vertical':
        gradient = np.linspace(0, 1, size)
        img[0] = gradient[:, None]
        img[1] = gradient[:, None] ** 2
        img[2] = 1 - gradient[:, None]
    elif direction == 'diagonal':
        x, y = np.meshgrid(np.linspace(0, 1, size), np.linspace(0, 1, size))
        img[0] = (x + y) / 2
        img[1] = np.abs(x - y)
        img[2] = 1 - (x + y) / 2
    else:  # radial
        center = size // 2
        y, x = np.ogrid[:size, :size]
        distance = np.sqrt((x - center)**2 + (y - center)**2)
        normalized_distance = distance / (size // 2)
        img[0] = normalized_distance
        img[1] = 1 - normalized_distance
        img[2] = np.sin(2 * np.pi * normalized_distance)
    
    return np.clip(img, 0, 1)

def create_texture_pattern(size):
    """Create texture-like patterns"""
    img = np.random.rand(3, size, size) * 0.3
    
    # Add some texture structure
    scale = np.random.randint(5, 20)
    for i in range(0, size, scale):
        for j in range(0, size, scale):
            end_i = min(i + scale, size)
            end_j = min(j + scale, size)
            color = np.random.rand(3)
            img[:, i:end_i, j:end_j] += color[:, None, None] * 0.7
    
    return np.clip(img, 0, 1)

def create_mixed_pattern(size):
    """Create mixed patterns"""
    # Combine multiple pattern types
    img1 = create_geometric_pattern(size)
    img2 = create_noise_pattern(size)
    img3 = create_gradient_pattern(size)
    
    weights = np.random.dirichlet([1, 1, 1])
    img = weights[0] * img1 + weights[1] * img2 + weights[2] * img3
    
    return np.clip(img, 0, 1)

# Generate synthetic data
print("Generating synthetic image-like data...")
images, labels = generate_synthetic_image_data(num_samples=4000, image_size=224)

print(f"Generated {len(images)} synthetic images")
print(f"Image shape: {images.shape}")
print(f"Labels distribution: {np.bincount(labels)}")

# Convert to PyTorch tensors
images_tensor = torch.FloatTensor(images)
labels_tensor = torch.LongTensor(labels)

# Create train/test split
X_train, X_test, y_train, y_test = train_test_split(
    images_tensor, labels_tensor, test_size=0.2, random_state=42, stratify=labels_tensor
)

# Create datasets and data loaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 32  # Smaller batch size for memory efficiency
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nData preparation complete:")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Batch size: {batch_size}")

# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

category_names = ['Geometric', 'Noise', 'Gradient', 'Texture', 'Mixed']

for i in range(10):
    idx = i * (len(images) // 10)
    img = images[idx].transpose(1, 2, 0)  # CHW to HWC for matplotlib
    label = labels[idx]
    
    axes[i].imshow(img)
    axes[i].set_title(f'{category_names[label]} (Class {label})')
    axes[i].axis('off')

plt.suptitle('Sample Synthetic Images for Transfer Learning', fontsize=16)
plt.tight_layout()
plt.show()

print("Synthetic data visualization complete!")

#  Feature Engineering with PyTorch for Transfer Learning

## Leveraging Pre-trained Models for Feature Extraction

This notebook demonstrates **Transfer Learning** techniques for feature engineering, covering:

- **Pre-trained CNN models** (ResNet, VGG, DenseNet)
- **Feature extraction** from frozen backbones
- **Fine-tuning strategies** for domain adaptation
- **Multi-level feature extraction** from different layers
- **Feature comparison** across architectures
- **Domain adaptation** techniques

### Learning Objectives:
1. Understand transfer learning principles and benefits
2. Extract features from pre-trained models
3. Compare features from different architectural layers
4. Fine-tune models for specific domains
5. Evaluate transfer learning effectiveness

### 🔧 Pre-trained Models:
- **ResNet**: Residual connections for deep networks
- **VGG**: Simple but effective CNN architecture
- **DenseNet**: Dense connectivity patterns
- **EfficientNet**: Efficient scaling of CNNs
- **Vision Transformers**: Self-attention for images

### Applications:
- Image classification with limited data
- Medical image analysis
- Satellite image processing
- Object detection and segmentation
- Cross-domain feature transfer

---